# M.Tech Thesis — Cross-lingual Hindi→Bhojpuri Dependency Parser

**Systems:**
- **System A** — Zero-shot baseline (Hindi model on Bhojpuri)
- **System F** — High-quality fine-tuning on professor's Bhojpuri data
- **System G** — Exact alignment joint training
- **System H** — Syntax-Aware Cross-lingual Transfer (SACT) ← novel
- **System I** — Language-Agnostic SACT with adversarial GRL ← best novel

**Before running:**
1. Upload `bhojpuri_matched_transferred.conllu` and `hindi_matched.conllu` to Google Drive → `thesis_data/` folder.
2. Runtime → Change runtime type → **T4 GPU**.
3. Run cells in order: **1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13**.

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
    assert 'T4' in torch.cuda.get_device_name(0) or torch.cuda.get_device_properties(0).total_memory > 10e9, \
        'WARNING: Not a T4 — switch runtime to T4 GPU'
else:
    raise RuntimeError('No GPU — go to Runtime > Change runtime type > T4 GPU')

## Cell 2 — Install dependencies
Do NOT reinstall torch (Colab's version is fine). Do NOT downgrade numpy.

In [ ]:
%%capture
# Only install what trankit needs on top of Colab's existing packages
!pip install "transformers==4.36.2" tokenizers sentencepiece
!pip install trankit

## Cell 3 — Patch trankit (all fixes in one place)
Patches: Python 3.12 mutable dataclasses, numpy 2.x aliases, UD scorer cycles/roots, tpipeline assertion.

In [ ]:
import re, pathlib, sys

trankit_root = pathlib.Path('/usr/local/lib/python3.12/dist-packages/trankit')

# ── Fix 1: Python 3.12 mutable dataclass defaults ────────────────────────────
f = trankit_root / 'adapter_transformers/adapter_config.py'
txt = f.read_text()
if 'from dataclasses import dataclass, field' not in txt:
    txt = txt.replace('from dataclasses import dataclass',
                      'from dataclasses import dataclass, field')
# Fix the known multi-line mutable default (InvertibleAdapterConfig)
old = '    invertible_adapter: Optional[dict] = InvertibleAdapterConfig(\n        block_type="nice", non_linearity="relu", reduction_factor=2\n    )'
new = '    invertible_adapter: Optional[dict] = field(default_factory=lambda: InvertibleAdapterConfig(\n        block_type="nice", non_linearity="relu", reduction_factor=2\n    ))'
txt = txt.replace(old, new)
# Fix any remaining single-line XxxConfig() defaults
txt = re.sub(
    r'(\s+\w+:\s*[^\n=]+?)\s*=\s*(\w+Config\(\))',
    lambda m: f'{m.group(1)} = field(default_factory={m.group(2)[:-2]})',
    txt
)
f.write_text(txt)
print('Fix 1 OK: adapter_config.py')

# ── Fix 2: numpy 2.x removed np.bool / np.int / np.float aliases ─────────────
count = 0
for p in trankit_root.rglob('*.py'):
    src = p.read_text()
    new = src
    new = re.sub(r'\bnp\.bool\b(?!_)',    'np.bool_',     new)
    new = re.sub(r'\bnp\.int\b(?!_)',     'np.int_',      new)
    new = re.sub(r'\bnp\.float\b(?!_)',   'np.float64',   new)
    new = re.sub(r'\bnp\.complex\b(?!_)', 'np.complex128',new)
    if new != src:
        p.write_text(new); count += 1
print(f'Fix 2 OK: numpy aliases patched in {count} files')

# ── Fix 3: UD scorer — allow multiple roots and cycles ────────────────────────
scorer = trankit_root / 'utils/scorers/conll18_ud_eval.py'
txt = scorer.read_text()
for bad in ['raise UDError("There are multiple roots in a sentence")',
            'raise UDError("There is a cycle in a sentence")']:
    txt = txt.replace(bad, 'pass  # patched: allow annotation artefacts')
scorer.write_text(txt)
print('Fix 3 OK: UD scorer (roots/cycles)')

# ── Fix 4: tpipeline assertion — allow local directory as embedding name ──────
tp = trankit_root.parent / 'trankit/tpipeline.py'
txt = tp.read_text()
old = 'assert master_config.embedding_name in supported_embeddings,'
new = ('assert master_config.embedding_name in supported_embeddings or '
       '__import__("os").path.isdir(master_config.embedding_name),')
if old in txt and 'isdir' not in txt:
    tp.write_text(txt.replace(old, new))
    print('Fix 4 OK: tpipeline.py assertion')
else:
    print('Fix 4 already applied')

# ── Verify ────────────────────────────────────────────────────────────────────
for mod in list(sys.modules):
    if 'trankit' in mod: del sys.modules[mod]
from trankit import TPipeline
print('\ntrankit import OK — all patches applied, ready to train')

## Cell 4 — Clone repository

In [ ]:
import os

REPO = 'https://github.com/Sansgithub22/mtechthesis4biaffinemultilingual-parser.git'
WORK = '/content/mtechthesis4biaffinemultilingual-parser'

if not os.path.exists(WORK):
    !git clone {REPO} {WORK}
else:
    !cd {WORK} && git pull

os.chdir(WORK)
print('Working dir:', os.getcwd())
!ls

## Cell 5 — Mount Google Drive and copy data files
Needs `bhojpuri_matched_transferred.conllu` and `hindi_matched.conllu` in Drive → `thesis_data/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import shutil, os

DRIVE_DATA = '/content/drive/MyDrive/thesis_data'
WORK       = '/content/mtechthesis4biaffinemultilingual-parser'

print('Files in Drive thesis_data:')
try:
    print(os.listdir(DRIVE_DATA))
except FileNotFoundError:
    raise FileNotFoundError(
        'thesis_data folder not found in Google Drive. '
        'Create it and upload the two .conllu files.')

for fname in ['bhojpuri_matched_transferred.conllu', 'hindi_matched.conllu']:
    src = f'{DRIVE_DATA}/{fname}'
    dst = f'{WORK}/{fname}'
    if not os.path.exists(src):
        print(f'NOT FOUND in Drive: {fname}')
    elif not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'Copied: {fname}  ({os.path.getsize(dst)/1e6:.1f} MB)')
    else:
        print(f'Already present: {fname}  ({os.path.getsize(dst)/1e6:.1f} MB)')

## Cell 6 — Download UD treebank data (Hindi HDTB + Bhojpuri BHTB test)

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')

!mkdir -p data_files/hindi data_files/bhojpuri data_files/sysf

HINDI_BASE = 'https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/master'
for split in ['train', 'dev', 'test']:
    dst = f'data_files/hindi/hi_hdtb-ud-{split}.conllu'
    if not os.path.exists(dst):
        !wget -q {HINDI_BASE}/hi_hdtb-ud-{split}.conllu -O {dst}
        print(f'Downloaded: {dst}')
    else:
        print(f'OK: {dst}')

BHO_BASE = 'https://raw.githubusercontent.com/UniversalDependencies/UD_Bhojpuri-BHTB/master'
dst = 'data_files/bhojpuri/bho_bhtb-ud-test.conllu'
if not os.path.exists(dst):
    !wget -q {BHO_BASE}/bho_bhtb-ud-test.conllu -O {dst}
    print(f'Downloaded: {dst}')
else:
    print(f'OK: {dst}')

print('\nAll data files ready.')

## Cell 7 — Train Hindi model (System A warm-start)
~1.5 hrs on T4. Saves to `checkpoints/trankit_hindi/`.

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')
!python3 train_trankit_hindi.py --epochs 60 --batch_size 32 --gpu

## Cell 8 — Save Hindi checkpoint to Drive
Run immediately after Cell 7 finishes — protects against Colab disconnect.

In [ ]:
import shutil, os

DRIVE_CKPT = '/content/drive/MyDrive/thesis_data/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)
src = '/content/mtechthesis4biaffinemultilingual-parser/checkpoints/trankit_hindi'
if os.path.exists(src):
    shutil.copytree(src, f'{DRIVE_CKPT}/trankit_hindi', dirs_exist_ok=True)
    print('Hindi checkpoint saved to Drive.')
else:
    print('ERROR: Hindi checkpoint not found.')

## Cell 9 — Train System F (high-quality fine-tuning on professor's data)
~2 hrs on T4.

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')
!python3 train_system_f.py --epochs 20 --batch_size 64 --gpu

## Cell 10 — Train System G (exact alignment joint training)
~1 hr on T4.

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')
!python3 train_system_g.py \
    --epochs 20 \
    --device cuda \
    --lambda_hi 0.3 \
    --lambda_align 0.5

## Cell 11 — Train System H (Syntax-Aware Cross-lingual Transfer) ← Novel
~1 hr on T4.

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')
!python3 train_system_h.py \
    --epochs 20 \
    --device cuda \
    --lambda_hi 0.3 \
    --lambda_cosine 0.4 \
    --lambda_arc 0.1 \
    --lambda_cts 0.6 \
    --warmup_epochs 3

## Cell 12 — Train System I (Language-Agnostic SACT) ← Best novel system
~1 hr on T4.

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')
!python3 train_system_i.py \
    --epochs 20 \
    --device cuda \
    --lambda_hi 0.3 \
    --lambda_cosine 0.4 \
    --lambda_arc 0.1 \
    --lambda_cts 0.6 \
    --lambda_adv 0.1 \
    --grl_alpha 1.0 \
    --warmup_epochs 3

## Cell 13 — Evaluate all systems on BHTB test set

In [ ]:
import os
os.chdir('/content/mtechthesis4biaffinemultilingual-parser')
!python3 evaluate_trankit.py --gpu

## Cell 14 — Save all checkpoints to Google Drive

In [ ]:
import shutil, os

DRIVE_CKPT = '/content/drive/MyDrive/thesis_data/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)

src = '/content/mtechthesis4biaffinemultilingual-parser/checkpoints'
if os.path.exists(src):
    shutil.copytree(src, DRIVE_CKPT, dirs_exist_ok=True)
    print('All checkpoints saved to Google Drive:')
    for root, dirs, files in os.walk(DRIVE_CKPT):
        for f in files:
            fp = os.path.join(root, f)
            print(f'  {fp}  ({os.path.getsize(fp)/1e6:.1f} MB)')
else:
    print('No checkpoints directory found.')